# 05 Experiment 2: Semantic Retrieval

This notebook evaluates how well different text embeddings perform in a semantic retrieval (nearest neighbors) task. We test the hypothesis that contextual embeddings provide more accurate semantic matches than sparse or static ones.

**Pipeline:**
1. Load saved embeddings (`.npy`).
2. Build a FAISS index for fast nearest-neighbor search.
3. Sample a set of query reviews.
4. Retrieve the top-k most similar reviews for each query.
5. Evaluate using Precision@k and Mean Reciprocal Rank (MRR) based on label matching.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import json
import numpy as np
import pandas as pd
sys.path.append('..')

from src.retrieval import evaluate_retrieval
from src.visualize import plot_metrics_comparison

## 1. Load Data and Ground Truth

In [ ]:
processed_path = '../data/processed/cleaned_reviews.parquet'
df = pd.read_parquet(processed_path)

# Using 'rating' as a proxy for ground truth category match in this sample. 
# In the full dataset with multiple product categories, use the category label instead.
labels_true = df['rating'].values
print(f"Loaded {len(df)} reviews for retrieval evaluation.")

## 2. Run Retrieval Evaluation for All Encoders

In [ ]:
embedding_dir = '../data/embeddings/'
results_dir = '../experiments/retrieval/'
figures_dir = '../experiments/figures/'
os.makedirs(results_dir, exist_ok=True)
os.makedirs(figures_dir, exist_ok=True)

encoders = ['tfidf', 'w2v', 'glove', 'sbert', 'bge']
all_metrics = {}

# Sample size for queries to speed up evaluation. 5000 is usually enough for stable metrics.
QUERY_SAMPLE_SIZE = 5000
K_VALUES = [5, 10, 50]

In [ ]:
for enc in encoders:
    emb_path = os.path.join(embedding_dir, f'{enc}.npy')
    if not os.path.exists(emb_path):
        print(f"Skipping {enc}, embeddings not found at {emb_path}")
        continue
        
    print(f"\n{'='*40}\nEvaluating {enc.upper()}\n{'='*40}")
    X = np.load(emb_path)
    
    metrics = evaluate_retrieval(
        X, 
        labels_true, 
        sample_size=QUERY_SAMPLE_SIZE, 
        k_values=K_VALUES
    )
    
    all_metrics[enc.upper()] = metrics
    
    # Save individual metrics
    with open(os.path.join(results_dir, f'{enc}_retrieval_metrics.json'), 'w') as f:
        json.dump(metrics, f, indent=4)
        
    print(f"Results for {enc.upper()}: {metrics}")

## 3. Compare Encoders

In [ ]:
if all_metrics:
    print("\nFinal Retrieval Metric Summary:")
    for enc, m in all_metrics.items():
        print(f"{enc:10s} - MRR: {m['mrr']:.4f}, P@10: {m['precision@10']:.4f}")
        
    # Save aggregated metrics
    with open(os.path.join(results_dir, 'all_retrieval_metrics.json'), 'w') as f:
        json.dump(all_metrics, f, indent=4)

    # Plot Bar Chart
    plot_metrics_comparison(
        all_metrics, 
        filename=os.path.join(figures_dir, 'retrieval_metrics_comparison.png')
    )
else:
    print("No encoders evaluated. Run the encode notebook first!")